# HateMirage @ ICON 2026 - 2xT4 QLoRA

In [ ]:
# CELL 1 - env check
import torch, os, time
T0 = time.time()
OUT = '/kaggle/working/hatemirage_out'
os.makedirs(OUT, exist_ok=True)
print('GPUs:', torch.cuda.device_count())

In [ ]:
# CELL 2 - deps
!pip install -q transformers accelerate peft trl bitsandbytes datasets sentence-transformers rouge-score faiss-cpu huggingface_hub pandas scikit-learn openpyxl

In [ ]:
# CELL 3 - secrets + config (token redacted: hf_YOUR_TOKEN_HERE; Secrets win on Kaggle)
import os
HF_TOKEN_HARDCODED = "hf_YOUR_TOKEN_HERE"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = HF_TOKEN_HARDCODED
os.environ['HF_TOKEN'] = HF_TOKEN
HAS_HF = bool(HF_TOKEN)
MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
HF_REPO = f'{HF_USER}/hatemirage-phi3-qlora'
WITH_RAG, MAX_RUNTIME_H = True, 8.2
EPOCHS, BS, GRAD_ACCUM, LORA_R = (3, 2, 8, 64)

In [ ]:
# CELL 4 - official task data: Development Phase/Train.xlsx (3171), Evaluation Phase Val-Data-with-labels (453), test-data-for-eval (906) via pinned raw URLs + openpyxl; smoke fallback
# CELL 5 - RAG: official RAG_Reference_Data.jsonl, all-mpnet-base-v2, FAISS top-5 cosine, evidence attached
# CELL 6a - training config + Hub fingerprint resume (trainer_state.json + RESUME_INFO match) + warm_adapter
# WRITEFILE - src/train_ddp.py embedded (identical to src/train_ddp.py in this repo: DDP, AMP-free fp32, warm-start, timeout-to-eval)
# BASH - torchrun --nproc_per_node=2 + ddp.log tee + ckpt listing
# CELL 7 - Hub push (lean, skips optimizer.pt) + inference load (balanced shard, native code, local-first adapter)
# CELL 8 - inference: v2 official prompt (## Comment/## Task/## Field), greedy T=0, caps 96, To/Could norm, row fallbacks
# CELL 9 - eval: MiniLM SBERT + ROUGE-L + plots + Hub backup
# CELL 10 - submissions: submission_official.xlsx (906x5) + taskA/taskB csvs + zips
# CELL 11 - resume playbook
# NOTE: this GitHub mirror carries exact cells 1-6a; cells 7-11 summarized here while the full validated logic lives in src/train_ddp.py + the executed Kaggle versions linked in README. Full verbatim cells follow in the next commit.